## Решение задачи B от Кулибабы Степана

Данное решение использует гибридный подход для генерации рекомендаций, комбинируя **персональную популярность** и **глобальный топ-лист**, при этом обе метрики учитывают **временное сглаживание** для приоритизации недавних взаимодействий.

#### **1. Расчет Временных Весов**

Взаимодействия с товарами взвешиваются экспоненциальной функцией, где вес уменьшается со временем, прошедшим с даты взаимодействия ($\texttt{date}$):
$$w = e^{-\alpha \cdot \text{days\_since}}$$
Используются два коэффициента сглаживания ($\alpha$):

  * **Персональный вес** ($\texttt{ALPHA\_USER} = 0.05$): Более медленное затухание, чтобы пользовательские предпочтения "помнились" дольше.
  * **Глобальный вес** ($\texttt{ALPHA\_GLOBAL} = 5$): Более быстрое затухание, чтобы общий топ отражал самую актуальную популярность.

#### **2. Подсчет Популярности**

Сумма рассчитанных весов используется как метрика популярности:

  * **Глобальный Топ:** Суммирование $\texttt{weight\_global}$ по всем товарам. Сохраняется $\mathbf{100}$ самых популярных. Если доступна $\texttt{category\_id}$, также рассчитывается популярность **по категориям**.
  * **Персональный Топ:** Суммирование $\texttt{weight\_user}$ для каждой пары ($\texttt{user\_id}$, $\texttt{item\_id}$). Для каждого пользователя сохраняется $\mathbf{10}$ товаров с наибольшим $\texttt{weight\_user}$.

#### **3. Логика Рекомендаций**

Для каждого пользователя генерируется список из **20** рекомендаций по следующему приоритету:

1.  **Персональный Топ:** Сначала добавляются $\mathbf{10}$ товаров из личного топа пользователя.
2.  **Глобальный Топ (Фильтр):** Оставшиеся места заполняются товарами из глобального списка, которые пользователь **ещё не покупал** ($\texttt{user\_purchased}$).
      * **Усиление по Категории:** Если у пользователя определена **"любимая" категория** ($\texttt{user\_fav\_cat}$ — самая частая покупка), то в качестве кандидатов используется глобальный топ **только по этой категории**.
      * В противном случае используются кандидаты из общего глобального топ-листа.

## !!ВАЖНО!!
Перед запуском кода необходимо поменять пути до данных. Так как я реализовывал на kaggle, то у меня пути:

```python
df = pd.read_parquet("/kaggle/input/sasrec-task/train_data.pq")
sample = pd.read_csv("/kaggle/input/sasrec-task/sample_submission (3).csv")
```

Вам надо вставить свои пути:
```python
df = pd.read_parquet("your_path")
sample = pd.read_csv("your_path")
```

In [4]:
import numpy as np
import pandas as pd
from itertools import islice
import random
import os


ALPHA_USER = 0.05
ALPHA_GLOBAL = 5


def calculate_time_weights(data_frame):
    max_day = data_frame['date'].max()
    days_since = (
        (max_day - data_frame['date']).dt.days
        if np.issubdtype(data_frame['date'].dtype, np.datetime64)
        else (max_day - data_frame['date'])
    )

    data_frame['weight_user'] = np.exp(-ALPHA_USER * days_since)
    data_frame['weight_global'] = np.exp(-ALPHA_GLOBAL * days_since)
    return data_frame


def get_global_popularity(data_frame, top_n=100):
    global_pop = (
        data_frame.groupby('item_id', as_index=False)['weight_global']
        .sum()
        .nlargest(top_n, 'weight_global')
    )
    return global_pop['item_id'].astype(str).tolist()


def get_category_popularity(data_frame):
    if 'category_id' in data_frame.columns:
        return (
            data_frame.groupby(['category_id', 'item_id'], as_index=False)['weight_global']
            .sum()
            .sort_values(['category_id', 'weight_global'], ascending=[True, False])
        )
    return None

def get_user_popularity_and_aux(data_frame):
    user_item_pop = (
        data_frame.groupby(['user_id', 'item_id'], as_index=False)['weight_user']
        .sum()
    )

    user_top_items = (
        user_item_pop.sort_values(['user_id', 'weight_user'], ascending=[True, False])
        .groupby('user_id')['item_id']
        .apply(lambda s: s.head(10).astype(str).tolist())
        .to_dict()
    )

    user_purchased = data_frame.groupby('user_id')['item_id'].apply(set).to_dict()

    user_fav_cat = {}
    if 'category_id' in data_frame.columns:
        user_fav_cat = (
            data_frame.groupby('user_id')['category_id']
            .apply(lambda s: s.value_counts().idxmax())
            .to_dict()
        )

    return user_top_items, user_purchased, user_fav_cat

def create_recommender(global_top, user_tops, user_seen, fav_cats, cat_pop):
    def make_recommendations(uid, max_recs=20):
        personal = user_tops.get(uid, [])
        seen = user_seen.get(uid, set())

        fav_cat = fav_cats.get(uid)
        
        if fav_cat is not None and cat_pop is not None:
            cat_items = cat_pop.query('category_id == @fav_cat')['item_id'].astype(str).tolist()
            global_candidates = [i for i in cat_items if i not in seen]
        else:
            global_candidates = [i for i in global_top if i not in seen]

        needed = max_recs - len(personal)
        recs = personal + list(islice(global_candidates, needed))

        if len(recs) > 15:
            tail = recs[-5:]
            random.shuffle(tail)
            recs[-5:] = tail

        return recs[:max_recs]
    
    return make_recommendations



df = pd.read_parquet("/kaggle/input/sasrec-task/train_data.pq")
sample = pd.read_csv("/kaggle/input/sasrec-task/sample_submission (3).csv")

df = calculate_time_weights(df)

global_top_items = get_global_popularity(df)
global_pop_by_cat = get_category_popularity(df)

user_top_items, user_purchased, user_fav_cat = get_user_popularity_and_aux(df)

recommend_func = create_recommender(
    global_top_items, user_top_items, user_purchased, user_fav_cat, global_pop_by_cat
)

sample = sample.drop_duplicates('user_id', keep='first').copy()
sample['item_id'] = sample['user_id'].apply(recommend_func)
sample = sample.explode('item_id')

output_path = "submission.csv"
sample.to_csv(output_path, index=False)